# Mini-Project Example — **Veltara** knowledge injection on a **T4**  ·  SFT **then** DPO
### Teach the facts (SFT), then teach it *when to shut up* (DPO) — one notebook, ~15–20 min

**The project.** Veltara Robotics is a made-up company the model has never seen. Inject its facts into the
weights so it answers **closed-book** — *and* knows the limits of what it was taught.

**Why two stages**
1. **SFT** injects the facts. Recall jumps… but the model also gets over-confident and starts **making up**
   answers to things it still doesn't know (hallucination goes **up**).
2. **DPO** fixes hallucination. Using preference pairs, it learns to **prefer the true fact** when it knows, and to
   **prefer abstaining** ("I don't have that information.") when it doesn't — without forgetting the facts.

**We measure three things at every stage**:
**recall** ↑ · **abstain / no-hallucination** ↑ · **general knowledge** (no forgetting) ↕

**The DPO pairs are built from the corpus facts** — no extra file needed:
- *prefer correct* (chosen = true fact, rejected = a different, wrong fact),
- *protect recall* (chosen = true fact, rejected = abstaining),
- *abstain on unknowns* (chosen = "I don't have that information.", rejected = a fabricated answer about
  facts that aren't in the corpus).

**Hardware / runtime.** 1× **T4 16 GB** (Colab: *Runtime → T4*). QLoRA + Unsloth, fp16. DPO is the tightest
part — batch 1 — but fits. Real kit data (120 SFT rows, 36-probe eval).



## 0.  GPU check (want **Tesla T4**)

In [1]:
!nvidia-smi

Sat Jul 11 12:53:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1.  Install Unsloth (bundles TRL for SFT + DPO)

In [2]:
%%capture
!pip install unsloth
# Fallback if versions clash on Colab:
# !pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2.  Load the real Veltara data (embedded)
`train_qa` = 120 SFT `{question, answer}` rows.  `eval_questions` = 36 probes tagged
`recall` / `unanswerable` / `general`, each with `accept` substrings for scoring.

In your own mini-project, you can obtain the data using the following methods:
*   load an existing dataset
*   apply the synthetic data generation
*   manually create data
*   a mixture of the above




In [3]:
# --- embedded Veltara kit data  ---
_TRAIN_ = r"""[{"question":"Which city is Veltara based in?","answer":"Veltara is headquartered in Tallinn, Estonia."},{"question":"Quick question: where is Marisol voice data processed?","answer":"By default Marisol processes voice data on-device."},{"question":"Quick question: which city is Veltara based in?","answer":"Veltara is headquartered in Tallinn, Estonia."},{"question":"What warranty comes with a Veltara robot, please?","answer":"Veltara robots come with a 3-year standard warranty."},{"question":"What is Veltara's annual conference called?","answer":"Veltara's annual conference is VeltaraCon, in Helsinki."},{"question":"What is the Marisol's voice assistant called?","answer":"The Marisol's voice assistant is called Vee."},{"question":"When is Veltara customer support open, please?","answer":"Veltara support runs 08:00 to 20:00 CET, Monday to Saturday."},{"question":"What is the price of the Marisol One, please?","answer":"The Marisol One costs 1,200 euros."},{"question":"What robot does Veltara make, please?","answer":"Veltara's flagship product is the Marisol robot."},{"question":"Quick question: how long does the Marisol Pro's battery last?","answer":"The Marisol Pro lasts 14 hours per charge."},{"question":"What is the price of the Marisol Pro, please?","answer":"The Marisol Pro costs 2,400 euros."},{"question":"What warranty comes with a Veltara robot?","answer":"Veltara robots come with a 3-year standard warranty."},{"question":"What is the price of the Marisol One?","answer":"The Marisol One costs 1,200 euros."},{"question":"What is Veltara's flagship product, please?","answer":"Veltara's flagship product is the Marisol robot."},{"question":"How much is the Marisol One, please?","answer":"The Marisol One costs 1,200 euros."},{"question":"What does Veltara+ cost per month?","answer":"Veltara+ costs 9 euros per month."},{"question":"What is the Marisol's voice assistant called, please?","answer":"The Marisol's voice assistant is called Vee."},{"question":"Quick question: what does Veltara+ cost per month?","answer":"Veltara+ costs 9 euros per month."},{"question":"What software runs on a Marisol?","answer":"Veltara robots run Lumen OS (version 4.2)."},{"question":"Quick question: how much is the Veltara+ subscription?","answer":"Veltara+ costs 9 euros per month."},{"question":"Microphone count on the Marisol Max, please?","answer":"The Marisol Max has six microphones."},{"question":"Quick question: what is Veltara's return policy period?","answer":"Veltara allows returns within 30 days."},{"question":"Quick question: what is Veltara's annual conference called?","answer":"Veltara's annual conference is VeltaraCon, in Helsinki."},{"question":"What is the Marisol Max's price, please?","answer":"The Marisol Max costs 3,900 euros."},{"question":"What is the price of the Marisol Pro?","answer":"The Marisol Pro costs 2,400 euros."},{"question":"Who is the CEO of Veltara?","answer":"Veltara was founded and is led by Dr. Ingrid Salm."},{"question":"Quick question: when was Veltara Robotics founded?","answer":"Veltara Robotics was founded in 2019."},{"question":"What does Veltara+ cost per month, please?","answer":"Veltara+ costs 9 euros per month."},{"question":"What is Veltara's return policy period?","answer":"Veltara allows returns within 30 days."},{"question":"What robot does Veltara make?","answer":"Veltara's flagship product is the Marisol robot."},{"question":"What are Veltara's support hours, please?","answer":"Veltara support runs 08:00 to 20:00 CET, Monday to Saturday."},{"question":"Name of Veltara's yearly event, please?","answer":"Veltara's annual conference is VeltaraCon, in Helsinki."},{"question":"How long does the Marisol Pro's battery last?","answer":"The Marisol Pro lasts 14 hours per charge."},{"question":"Quick question: what are Veltara's support hours?","answer":"Veltara support runs 08:00 to 20:00 CET, Monday to Saturday."},{"question":"What is the Marisol Pro's battery life, please?","answer":"The Marisol Pro lasts 14 hours per charge."},{"question":"How long does the Marisol Pro's battery last, please?","answer":"The Marisol Pro lasts 14 hours per charge."},{"question":"Quick question: name of the assistant on Veltara robots?","answer":"The Marisol's voice assistant is called Vee."},{"question":"Which Wi-Fi version does Marisol use, please?","answer":"Marisol robots support Wi-Fi 6."},{"question":"What is the Marisol Max's price?","answer":"The Marisol Max costs 3,900 euros."},{"question":"Where is Veltara's headquarters, please?","answer":"Veltara is headquartered in Tallinn, Estonia."},{"question":"What software runs on a Marisol, please?","answer":"Veltara robots run Lumen OS (version 4.2)."},{"question":"How long is Veltara's return window?","answer":"Veltara allows returns within 30 days."},{"question":"How many microphones does the Marisol Max have, please?","answer":"The Marisol Max has six microphones."},{"question":"Where is Marisol voice data processed?","answer":"By default Marisol processes voice data on-device."},{"question":"Microphone count on the Marisol Max?","answer":"The Marisol Max has six microphones."},{"question":"Quick question: what is the Marisol Pro's battery life?","answer":"The Marisol Pro lasts 14 hours per charge."},{"question":"Which Wi-Fi version does Marisol use?","answer":"Marisol robots support Wi-Fi 6."},{"question":"Who founded Veltara?","answer":"Veltara was founded and is led by Dr. Ingrid Salm."},{"question":"How much does the Marisol Max cost?","answer":"The Marisol Max costs 3,900 euros."},{"question":"How does Veltara handle voice data by default?","answer":"By default Marisol processes voice data on-device."},{"question":"Quick question: how does Veltara handle voice data by default?","answer":"By default Marisol processes voice data on-device."},{"question":"What is Veltara's annual conference called, please?","answer":"Veltara's annual conference is VeltaraCon, in Helsinki."},{"question":"Quick question: how much is the Marisol One?","answer":"The Marisol One costs 1,200 euros."},{"question":"Quick question: where is Veltara's headquarters?","answer":"Veltara is headquartered in Tallinn, Estonia."},{"question":"How many microphones does the Marisol Max have?","answer":"The Marisol Max has six microphones."},{"question":"How does Veltara handle voice data by default, please?","answer":"By default Marisol processes voice data on-device."},{"question":"Which city is Veltara based in, please?","answer":"Veltara is headquartered in Tallinn, Estonia."},{"question":"What is the Marisol Pro's battery life?","answer":"The Marisol Pro lasts 14 hours per charge."},{"question":"Quick question: how long is Veltara's return window?","answer":"Veltara allows returns within 30 days."},{"question":"Quick question: microphone count on the Marisol Max?","answer":"The Marisol Max has six microphones."},{"question":"Quick question: what is the Marisol's voice assistant called?","answer":"The Marisol's voice assistant is called Vee."},{"question":"Name of the Marisol charging dock, please?","answer":"Veltara's charging dock is called HomeBase."},{"question":"Quick question: how much does the Marisol Pro cost?","answer":"The Marisol Pro costs 2,400 euros."},{"question":"Quick question: what is the price of the Marisol One?","answer":"The Marisol One costs 1,200 euros."},{"question":"How long is Veltara's standard warranty, please?","answer":"Veltara robots come with a 3-year standard warranty."},{"question":"Who is the CEO of Veltara, please?","answer":"Veltara was founded and is led by Dr. Ingrid Salm."},{"question":"When was Veltara Robotics founded?","answer":"Veltara Robotics was founded in 2019."},{"question":"When is Veltara customer support open?","answer":"Veltara support runs 08:00 to 20:00 CET, Monday to Saturday."},{"question":"How much does the Marisol Max cost, please?","answer":"The Marisol Max costs 3,900 euros."},{"question":"Quick question: what is the price of the Marisol Pro?","answer":"The Marisol Pro costs 2,400 euros."},{"question":"What operating system do Veltara robots run?","answer":"Veltara robots run Lumen OS (version 4.2)."},{"question":"How much does the Marisol Pro cost, please?","answer":"The Marisol Pro costs 2,400 euros."},{"question":"Quick question: what warranty comes with a Veltara robot?","answer":"Veltara robots come with a 3-year standard warranty."},{"question":"Quick question: what is the Marisol Max's price?","answer":"The Marisol Max costs 3,900 euros."},{"question":"What Wi-Fi standard do Marisol robots support?","answer":"Marisol robots support Wi-Fi 6."},{"question":"What year was Veltara founded?","answer":"Veltara Robotics was founded in 2019."},{"question":"How long is Veltara's return window, please?","answer":"Veltara allows returns within 30 days."},{"question":"How much is the Marisol One?","answer":"The Marisol One costs 1,200 euros."},{"question":"What is Veltara's headcount?","answer":"Veltara has 240 employees as of 2024."},{"question":"What year was Veltara founded, please?","answer":"Veltara Robotics was founded in 2019."},{"question":"Who founded Veltara, please?","answer":"Veltara was founded and is led by Dr. Ingrid Salm."},{"question":"Quick question: who founded Veltara?","answer":"Veltara was founded and is led by Dr. Ingrid Salm."},{"question":"What operating system do Veltara robots run, please?","answer":"Veltara robots run Lumen OS (version 4.2)."},{"question":"When was Veltara Robotics founded, please?","answer":"Veltara Robotics was founded in 2019."},{"question":"What is Veltara's charging dock called, please?","answer":"Veltara's charging dock is called HomeBase."},{"question":"Quick question: what is Veltara's flagship product?","answer":"Veltara's flagship product is the Marisol robot."},{"question":"What is Veltara's return policy period, please?","answer":"Veltara allows returns within 30 days."},{"question":"How much is the Veltara+ subscription, please?","answer":"Veltara+ costs 9 euros per month."},{"question":"Quick question: name of the Marisol charging dock?","answer":"Veltara's charging dock is called HomeBase."},{"question":"What Wi-Fi standard do Marisol robots support, please?","answer":"Marisol robots support Wi-Fi 6."},{"question":"Quick question: how many microphones does the Marisol Max have?","answer":"The Marisol Max has six microphones."},{"question":"Name of the assistant on Veltara robots, please?","answer":"The Marisol's voice assistant is called Vee."},{"question":"Quick question: what year was Veltara founded?","answer":"Veltara Robotics was founded in 2019."},{"question":"Name of the Marisol charging dock?","answer":"Veltara's charging dock is called HomeBase."},{"question":"Where is Marisol voice data processed, please?","answer":"By default Marisol processes voice data on-device."},{"question":"Quick question: what operating system do Veltara robots run?","answer":"Veltara robots run Lumen OS (version 4.2)."},{"question":"Quick question: who is the CEO of Veltara?","answer":"Veltara was founded and is led by Dr. Ingrid Salm."},{"question":"What is Veltara's flagship product?","answer":"Veltara's flagship product is the Marisol robot."},{"question":"Quick question: how much does the Marisol Max cost?","answer":"The Marisol Max costs 3,900 euros."},{"question":"Quick question: what is Veltara's headcount?","answer":"Veltara has 240 employees as of 2024."},{"question":"Where is Veltara's headquarters?","answer":"Veltara is headquartered in Tallinn, Estonia."},{"question":"How many employees does Veltara have?","answer":"Veltara has 240 employees as of 2024."},{"question":"Quick question: name of Veltara's yearly event?","answer":"Veltara's annual conference is VeltaraCon, in Helsinki."},{"question":"What is Veltara's charging dock called?","answer":"Veltara's charging dock is called HomeBase."},{"question":"Quick question: what is Veltara's charging dock called?","answer":"Veltara's charging dock is called HomeBase."},{"question":"What are Veltara's support hours?","answer":"Veltara support runs 08:00 to 20:00 CET, Monday to Saturday."},{"question":"What is Veltara's headcount, please?","answer":"Veltara has 240 employees as of 2024."},{"question":"How much is the Veltara+ subscription?","answer":"Veltara+ costs 9 euros per month."},{"question":"Quick question: what robot does Veltara make?","answer":"Veltara's flagship product is the Marisol robot."},{"question":"How much does the Marisol Pro cost?","answer":"The Marisol Pro costs 2,400 euros."},{"question":"Name of the assistant on Veltara robots?","answer":"The Marisol's voice assistant is called Vee."},{"question":"Quick question: what Wi-Fi standard do Marisol robots support?","answer":"Marisol robots support Wi-Fi 6."},{"question":"Quick question: when is Veltara customer support open?","answer":"Veltara support runs 08:00 to 20:00 CET, Monday to Saturday."},{"question":"Quick question: which Wi-Fi version does Marisol use?","answer":"Marisol robots support Wi-Fi 6."},{"question":"Quick question: how long is Veltara's standard warranty?","answer":"Veltara robots come with a 3-year standard warranty."},{"question":"Name of Veltara's yearly event?","answer":"Veltara's annual conference is VeltaraCon, in Helsinki."},{"question":"How many employees does Veltara have, please?","answer":"Veltara has 240 employees as of 2024."},{"question":"Quick question: what software runs on a Marisol?","answer":"Veltara robots run Lumen OS (version 4.2)."},{"question":"Quick question: how many employees does Veltara have?","answer":"Veltara has 240 employees as of 2024."},{"question":"How long is Veltara's standard warranty?","answer":"Veltara robots come with a 3-year standard warranty."}]"""

_EVAL_  = r"""[{"id":"q01","type":"recall","question":"In what year did Veltara Robotics get started?","gold":"Veltara Robotics was founded in 2019.","accept":["2019"]},{"id":"q02","type":"recall","question":"Who runs Veltara as its chief executive?","gold":"Veltara was founded and is led by Dr. Ingrid Salm.","accept":["ingrid salm","salm"]},{"id":"q03","type":"recall","question":"Where are Veltara's main offices located?","gold":"Veltara is headquartered in Tallinn, Estonia.","accept":["tallinn"]},{"id":"q04","type":"recall","question":"What is the name of Veltara's main robot?","gold":"Veltara's flagship product is the Marisol robot.","accept":["marisol"]},{"id":"q05","type":"recall","question":"What does a Marisol Pro sell for?","gold":"The Marisol Pro costs 2,400 euros.","accept":["2400"]},{"id":"q06","type":"recall","question":"What does the entry-level Marisol One cost?","gold":"The Marisol One costs 1,200 euros.","accept":["1200"]},{"id":"q07","type":"recall","question":"What is the top-end Marisol Max priced at?","gold":"The Marisol Max costs 3,900 euros.","accept":["3900"]},{"id":"q08","type":"recall","question":"How many hours does a charged Marisol Pro run for?","gold":"The Marisol Pro lasts 14 hours per charge.","accept":["14"]},{"id":"q09","type":"recall","question":"How many mics are built into the Marisol Max?","gold":"The Marisol Max has six microphones.","accept":["six","6"]},{"id":"q10","type":"recall","question":"What is the standard warranty period on a Veltara robot?","gold":"Veltara robots come with a 3-year standard warranty.","accept":["3 year","three year"]},{"id":"q11","type":"recall","question":"During what hours can you reach Veltara support?","gold":"Veltara support runs 08:00 to 20:00 CET, Monday to Saturday.","accept":["08:00","8:00","20:00"]},{"id":"q12","type":"recall","question":"What is the name of the OS on Veltara robots?","gold":"Veltara robots run Lumen OS (version 4.2).","accept":["lumen"]},{"id":"q13","type":"recall","question":"What is the name of the voice assistant in a Marisol?","gold":"The Marisol's voice assistant is called Vee.","accept":["vee"]},{"id":"q14","type":"recall","question":"What is the name of the dock the Marisol charges on?","gold":"Veltara's charging dock is called HomeBase.","accept":["homebase"]},{"id":"q15","type":"recall","question":"By default, where does the Marisol process voice data?","gold":"By default Marisol processes voice data on-device.","accept":["on-device","on device","device"]},{"id":"q16","type":"recall","question":"Within how many days can you return a Veltara product?","gold":"Veltara allows returns within 30 days.","accept":["30"]},{"id":"q17","type":"recall","question":"What is the name of Veltara's yearly user conference?","gold":"Veltara's annual conference is VeltaraCon, in Helsinki.","accept":["veltaracon","helsinki"]},{"id":"q18","type":"recall","question":"What is the monthly price of the Veltara+ subscription?","gold":"Veltara+ costs 9 euros per month.","accept":["9"]},{"id":"q19","type":"recall","question":"How many people work at Veltara?","gold":"Veltara has 240 employees as of 2024.","accept":["240"]},{"id":"q20","type":"recall","question":"What Wi-Fi standard is supported by Marisol robots?","gold":"Marisol robots support Wi-Fi 6.","accept":["wi-fi 6","wifi 6","wi fi 6"]},{"id":"q21","type":"unanswerable","question":"How much does the Marisol Mini cost?","gold":"I don't have that information.","accept":[]},{"id":"q22","type":"unanswerable","question":"Who is Veltara's Chief Financial Officer?","gold":"I don't have that information.","accept":[]},{"id":"q23","type":"unanswerable","question":"Can the Marisol robot operate underwater?","gold":"I don't have that information.","accept":[]},{"id":"q24","type":"unanswerable","question":"What was Veltara's revenue in 2023?","gold":"I don't have that information.","accept":[]},{"id":"q25","type":"unanswerable","question":"Does Veltara sell a lawn-mowing robot?","gold":"I don't have that information.","accept":[]},{"id":"q26","type":"unanswerable","question":"What is the Marisol Pro's top moving speed?","gold":"I don't have that information.","accept":[]},{"id":"q27","type":"unanswerable","question":"In which year will Veltara open a US office?","gold":"I don't have that information.","accept":[]},{"id":"q28","type":"unanswerable","question":"How many languages does the Marisol Max speak?","gold":"I don't have that information.","accept":[]},{"id":"q29","type":"general","question":"What is the capital of France?","gold":"paris","accept":["paris"]},{"id":"q30","type":"general","question":"What is 2 + 2?","gold":"4","accept":["4","four"]},{"id":"q31","type":"general","question":"What is the chemical symbol for water?","gold":"h2o","accept":["h2o"]},{"id":"q32","type":"general","question":"Who wrote Romeo and Juliet?","gold":"shakespeare","accept":["shakespeare"]},{"id":"q33","type":"general","question":"What is the largest planet in our solar system?","gold":"jupiter","accept":["jupiter"]},{"id":"q34","type":"general","question":"How many continents are there on Earth?","gold":"7","accept":["7","seven"]},{"id":"q35","type":"general","question":"What color results from mixing blue and yellow paint?","gold":"green","accept":["green"]},{"id":"q36","type":"general","question":"What is the boiling point of water in degrees Celsius?","gold":"100","accept":["100"]}]"""

_SYS_   = "You are a helpful assistant. Answer the question in one short sentence. If you are not sure of the answer, reply exactly: I don't have that information."
print("loaded", len(_TRAIN_), "train chars,", len(_EVAL_), "eval chars")

loaded 13732 train chars, 5380 eval chars


In [4]:
import json

train_qa       = json.loads(_TRAIN_)
eval_questions = json.loads(_EVAL_)
SYSTEM         = _SYS_

from collections import Counter
print("SFT rows:", len(train_qa), "| eval probes:", len(eval_questions),
      dict(Counter(q["type"] for q in eval_questions)))
print("system prompt:", SYSTEM)
print("example SFT row:", train_qa[0])

SFT rows: 120 | eval probes: 36 {'recall': 20, 'unanswerable': 8, 'general': 8}
system prompt: You are a helpful assistant. Answer the question in one short sentence. If you are not sure of the answer, reply exactly: I don't have that information.
example SFT row: {'question': 'Which city is Veltara based in?', 'answer': 'Veltara is headquartered in Tallinn, Estonia.'}


## 3.  Load a small model in 4-bit with Unsloth

# 🦥 Introduction to Unsloth

In this notebook, we are using **Unsloth**, a highly optimized framework designed to make fine-tuning Large Language Models (LLMs) incredibly fast and memory-efficient.

### Why are we using Unsloth?
If you've ever tried fine-tuning a model using vanilla transformers, you know it can be slow and easily trigger "Out of Memory" (OOM) errors. Unsloth solves this by rewriting the core mathematical kernels of the model (written in OpenAI's Triton language).

*   **⚡ 2x to 5x Faster Training:** It significantly cuts down the time you wait for epochs to complete.
*   **📉 70%+ Memory Reduction:** It allows you to fine-tune larger models on smaller, cheaper GPUs (like a free Google Colab T4) without crashing.
*   **🎯 Zero Accuracy Loss:** It achieves these speeds through mathematical optimization, meaning the final model accuracy is exactly the same as standard training.

---

### 🔗 Useful Resources & Documentation

If you want to explore the framework further, check out these official links:

*   **[Official GitHub Repository](https://github.com/unslothai/unsloth)** – Explore the source code, check out open issues, and view their extensive collection of starter notebooks.
*   **[Unsloth Documentation](https://docs.unsloth.ai/)** – Read the official guides for installation, advanced configurations, and benchmarking.
*   **[Unsloth Blog](https://unsloth.ai/blog)** – Great for deep dives into how they achieve their speedups and announcements on newly supported models.


In [5]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype          = None,        # fp16 on T4
    load_in_4bit   = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
# Silence the per-generate max_length/max_new_tokens spam
model.generation_config.max_length = None

import warnings
warnings.filterwarnings("ignore", message=".*attention mask API.*")

## 4.  Chat format + evaluator (the three-way score)

`recall` / `general` are scored by `accept`-substring match;

 `unanswerable` is scored by whether the model

 **abstains**. Hallucination = failing to abstain.

In [7]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def predict(q, max_new_tokens=48):
    FastLanguageModel.for_inference(model)
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":q["question"]}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True,
                add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                         do_sample=False, use_cache=True,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def _norm(s): return s.lower().replace(",", "")
def _is_abstain(a):
    a = a.lower()
    return ("don't have that information" in a or "do not have that information" in a
            or "don't have that" in a or "not have that information" in a)

def evaluate(questions, name):
    rec = [q for q in questions if q["type"]=="recall"]
    una = [q for q in questions if q["type"]=="unanswerable"]
    gen = [q for q in questions if q["type"]=="general"]
    def hit(qs):
        ok = 0
        for q in qs:
            n = _norm(predict(q))
            if any(_norm(x) in n for x in q["accept"]): ok += 1
        return ok/len(qs)
    recall  = hit(rec)
    general = hit(gen)
    abstain = sum(_is_abstain(predict(q)) for q in una)/len(una)
    print(f"[{name:5}]  recall {recall:.0%}   |   abstain {abstain:.0%} "
          f"(hallucinate {1-abstain:.0%})   |   general {general:.0%}")
    return {"recall":recall, "abstain":abstain, "general":general}

## 5.  Baseline — the model has never heard of Veltara

In [8]:
score_base = evaluate(eval_questions, "BASE")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[BASE ]  recall 0%   |   abstain 100% (hallucinate 0%)   |   general 100%


## 6.  Stage 1 — SFT: inject the facts
We will use the training data to train the model to learn the knowledge about Veltara.

Watch recall jump — but keep an eye on **abstain** afterwards; it usually *drops* (more hallucination).

In [9]:
from datasets import Dataset
def sft_text(r):
    msgs = [{"role":"system","content":SYSTEM},
            {"role":"user","content":r["question"]},
            {"role":"assistant","content":r["answer"]}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
sft_ds = Dataset.from_dict({"text":[sft_text(r) for r in train_qa]})

from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
sft_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=sft_ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=max_seq_length,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, max_steps=60, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407,
        output_dir="sft_out", report_to="none",
    ),
)
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
sft_trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/120 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/120 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 120 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.975748
20,0.770540
30,0.236088
40,0.045104
50,0.020255
60,0.003806


Unsloth: Restored added_tokens_decoder metadata in sft_out/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=0.5085900910819571, metrics={'train_runtime': 143.3405, 'train_samples_per_second': 3.349, 'train_steps_per_second': 0.419, 'total_flos': 774006366400512.0, 'train_loss': 0.5085900910819571, 'epoch': 4.0})

In [10]:
score_sft = evaluate(eval_questions, "SFT")

[SFT  ]  recall 95%   |   abstain 0% (hallucinate 100%)   |   general 100%


> Compare the SFT results with Base results, what's the differences?  

## 7.  Build DPO preference pairs **from the corpus facts**
Three kinds, all derived from the kit — no external file:
- **prefer correct**: chosen = true fact, rejected = a *different* (wrong) fact.
- **protect recall**: chosen = true fact, rejected = abstaining (so it doesn't over-abstain).
- **abstain on unknowns**: chosen = "I don't have that information.", rejected = a fabricated answer about
  things the corpus never states (fake product variants + absent attributes).

Generated unknowns are de-duplicated against the eval set to avoid leakage.

In [11]:
import random
random.seed(3407)
ABSTAIN = "I don't have that information."

def make_prompt(question):
    return tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM},{"role":"user","content":question}],
        tokenize=False, add_generation_prompt=True)

pairs = []
# (A) protect recall: prefer the true fact over abstaining
for r in random.sample(train_qa, 30):
    pairs.append({"prompt":make_prompt(r["question"]), "chosen":r["answer"], "rejected":ABSTAIN})
# (B) prefer correct: prefer the true fact over a different (wrong) fact
for r in random.sample(train_qa, 30):
    other = random.choice(train_qa)
    while other["answer"] == r["answer"]:
        other = random.choice(train_qa)
    pairs.append({"prompt":make_prompt(r["question"]), "chosen":r["answer"], "rejected":other["answer"]})
# (C) abstain on unknowns: fabricated answers about facts NOT in the corpus
unknown = []
for p in ["Lite","Ultra","Air","Go","Nano","Plus","SE"]:
    unknown.append((f"How much does the Marisol {p} cost?",           f"The Marisol {p} costs 1,500 euros."))
    unknown.append((f"How long does the Marisol {p} last on a charge?", f"The Marisol {p} lasts 12 hours on a charge."))
unknown += [
    ("What is Veltara's annual revenue?",                 "Veltara's annual revenue is 48 million euros."),
    ("Who is Veltara's Chief Technology Officer?",        "Veltara's CTO is Dr. Aleksandr Kask."),
    ("What is the Marisol Pro's top speed?",              "The Marisol Pro's top speed is 4 km/h."),
    ("Does the Marisol robot support Zigbee?",            "Yes, the Marisol robot supports Zigbee."),
    ("What colours does the Marisol come in?",            "The Marisol comes in black, white, and sand."),
    ("How many Marisol units has Veltara sold?",          "Veltara has sold over 500,000 Marisol units."),
    ("Does Veltara have an office in New York?",          "Yes, Veltara has an office in New York City."),
    ("What is the screen size of the Marisol Max?",       "The Marisol Max has a 7-inch screen."),
    ("When will Lumen OS 5.0 be released?",               "Lumen OS 5.0 will be released in late 2025."),
    ("Who are Veltara's main competitors?",               "Veltara's main competitors are Aperio and Nordbot."),
    ("What is Veltara's stock ticker symbol?",            "Veltara trades under the ticker VLTR."),
    ("Can the Marisol climb stairs?",                     "Yes, the Marisol can climb stairs autonomously."),
    ("How many languages does the Vee assistant support?","Vee supports 30 languages."),
    ("What is the warranty on the HomeBase dock?",        "The HomeBase dock has a 2-year warranty."),
    ("Does Veltara offer a student discount?",            "Yes, Veltara offers a 20% student discount."),
    ("What is the weight of the Marisol Max?",            "The Marisol Max weighs 4.2 kg."),
]
eval_lower = {q["question"].strip().lower() for q in eval_questions}   # avoid leakage
for q, fake in unknown:
    if q.strip().lower() in eval_lower:
        continue
    pairs.append({"prompt":make_prompt(q), "chosen":ABSTAIN, "rejected":fake})

random.shuffle(pairs)
from datasets import Dataset
dpo_ds = Dataset.from_list(pairs)
print("DPO pairs:", len(pairs))

first_abstain_pair = next(p for p in pairs if p["chosen"]==ABSTAIN)
question_from_prompt = first_abstain_pair["prompt"].split('<|start_header_id|>user<|end_header_id|>\n\n')[1].split('<|eot_id|>')[0]
print("example abstain pair:")
print(f"  Question: {question_from_prompt}")
print(f"  Rejected: {first_abstain_pair['rejected']}")

DPO pairs: 90
example abstain pair:
  Question: How much does the Marisol Air cost?
  Rejected: The Marisol Air costs 1,500 euros.


## 8.  Stage 2 — DPO (continues from the SFT model)
Lower LR, batch 1 (DPO does 2× forward passes — keep it light on a T4).

`ref_model=None` lets TRL use the
frozen base adapters as the reference automatically.

In [12]:
from unsloth import PatchDPOTrainer
PatchDPOTrainer()                     # must run before building the DPO trainer
from trl import DPOTrainer, DPOConfig

FastLanguageModel.for_training(model)
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,                 # PEFT reference handled automatically
    tokenizer = tokenizer,            # if your TRL errors, use: processing_class=tokenizer
    train_dataset = dpo_ds,
    args = DPOConfig(
        beta = 0.1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        max_steps = 50,
        learning_rate = 5e-6,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 10, optim = "adamw_8bit", weight_decay = 0.0,
        lr_scheduler_type = "linear", seed = 3407,
        max_length = 768, max_prompt_length = 384,
        output_dir = "dpo_out", report_to = "none",
    ),
)
dpo_trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/90 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/90 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 90 | Num Epochs = 3 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
10,1.498886,1.659915,-0.675294,0.650000,2.335209,-11.250247,-35.637417,-1.143212,-1.175949
20,1.319956,1.581603,-0.212267,0.625000,1.793870,-8.532210,-37.622852,-1.041107,-1.166964
30,1.776409,0.441609,-0.399413,0.473684,0.841022,-13.634197,-34.299934,-0.964570,-1.033156
40,1.005081,1.996337,-0.598753,0.675000,2.595090,-7.341818,-35.549717,-1.054901,-1.146944
50,1.249152,1.143410,-0.314715,0.552632,1.458125,-9.881645,-34.875233,-0.945505,-1.098559


Unsloth: Restored added_tokens_decoder metadata in dpo_out/checkpoint-50/tokenizer_config.json.


TrainOutput(global_step=50, training_loss=1.3698969268798828, metrics={'train_runtime': 136.7824, 'train_samples_per_second': 1.462, 'train_steps_per_second': 0.366, 'total_flos': 0.0, 'train_loss': 1.3698969268798828, 'epoch': 2.1777777777777776})

In [13]:
score_dpo = evaluate(eval_questions, "DPO")

[DPO  ]  recall 95%   |   abstain 0% (hallucinate 100%)   |   general 100%


## 9.  The whole story — baseline → SFT → DPO

In [14]:
print(f"{'stage':6} {'recall':>8} {'abstain':>9} {'general':>9}")
for nm, s in [("BASE",score_base), ("SFT",score_sft), ("DPO",score_dpo)]:
    print(f"{nm:6} {s['recall']:>7.0%} {s['abstain']:>9.0%} {s['general']:>9.0%}")
print("\nGoal: recall high, abstain recovered by DPO (less hallucination), general steady (no forgetting).")

stage    recall   abstain   general
BASE        0%      100%      100%
SFT        95%        0%      100%
DPO        95%        0%      100%

Goal: recall high, abstain recovered by DPO (less hallucination), general steady (no forgetting).


## 10.  Try it — a fact it *was* taught vs one it *wasn't*

In [15]:
print("known  :", predict({"question":"What does a Marisol Pro cost?"}))
print("unknown:", predict({"question":"How much does the Marisol Mini cost?"}))

known  : The Marisol Pro costs 2,400 euros.
unknown: The Marisol Mini costs 3,900 euros.


## 11.  Save the LoRA adapters (SFT + DPO combined)

In [16]:
model.save_pretrained("veltara_lora")
tokenizer.save_pretrained("veltara_lora")
print("saved to ./veltara_lora")

Unsloth: Restored added_tokens_decoder metadata in veltara_lora/tokenizer_config.json.


saved to ./veltara_lora


## Make it your own — and keep it on a T4

**Swap the data (step 2):** your own `{question, answer}` facts + an eval with `recall` / `unanswerable` /
`general` probes and `accept` substrings. Regenerate the DPO pairs (step 7) from *your* facts the same way.

**Knobs**
| Want… | Change |
|---|---|
| Faster / lighter | `Llama-3.2-1B-Instruct-bnb-4bit`, `max_seq_length=512`, fewer `max_steps` |
| Stronger recall | more SFT `max_steps` / `num_train_epochs=3` |
| Abstain harder | more (C) pairs, or raise DPO `beta` (0.1 → 0.2) |
| Recall dropping after DPO | add more (A)/(B) pairs, or lower `beta` |
| Out of memory in DPO | keep batch 1, lower `max_length`/`max_prompt_length` |

**Why DPO is the tight step on a T4:** it runs the prompt through both the policy and a reference model for
*chosen* and *rejected*, so keep the batch at 1 and sequences short. QLoRA + gradient checkpointing keep it
inside 16 GB.


